**Interactive Jupyter notebook implementing the interpolation, Iwasawa power series approximation, shifts of a Iwasawa power series and Newton slope extraction. Includes commented test runs.**  

By shifting an Iwasawa power series by $a \in p\mathbb{Z}_p$ one can find zeros in small balls around $a$. Evaluating at $T+a$ shifts zeros from $\alpha$ to $\alpha - a$.

*Example*: `p=3, shift=15, slopes=[3]` means a zero lies in the ball $15+p^3 \mathbb{Z}_p$.

In [1]:
def power_series_shift(D, p, i, shift, verb=0):
    r"""
    Computes the shifted Iwasawa power series associated to a quadratic Dirichlet character, 
    Newton slope extraction, and ramification classification.
        
    INPUT:
    - ``D`` -- Fundamental discriminant.
    - ``p`` -- Prime number p (odd). 
    - ``i`` -- Teichmüller twist i (0 <= i < p - 1).
    - ``shift``-- array of integers
    - ``verb`` -- verbosity: only characters with lambda >= verb are printed 
    
    OUTPUT: 
    - ``fp`` -- Approximating polynomial of the (unshifted) Iwasawa power series for each a in shift.
    - ``lambda_inv`` -- Lambda-invariant of the power series.
    - ``v0`` -- Valuation of the constant coefficient of the power series.

    AUTHOR: Heiko Knospe, 2026
    """
    
    K = Qp(p, prec=50)
    v = K.valuation()
    R = PolynomialRing(K, 'x')

    if not Integer(D).is_fundamental_discriminant():
        print("Not a fundamental discriminant")
        return None, None, None

    if (D>0 and i%2==1) or (D<0 and i%2==0):
        print("Trivial p-adic L-function")
        return None, None, None

    cond = abs(D)
        
    if gcd(cond, p) > 1:
        print("Not supported. gcd(|D|, p) > 1.")
        return None, None, None

    if i not in range(0, p - 1):
        print("Teichmuller twist invalid")
        return None, None, None

    if not(isinstance(shift, list) and shift):
        print("Shift array invalid")
        return None, None, None

    for sh in shift:
        if v(sh)<=0:
            print("Shift array invalid") # shifts must have positive valuation
            return None, None, None

    chip = kronecker(D, p)
    
    if i == 1 and chip == 1:
        print(f"p={p}, twist={i}, D={D} ==> Trivial zero case")
        return None, None, None
      
    # Compute special values of the p-adic L-function
    # Variable: T = (1 + cond * p)^(-j + 1) - 1
    points = [
        (K((1 + cond * p)**(-j + 1) - 1), pari.lfun(D, 1 - j) * (1 - chip * p**(j - 1))) 
        for j in range(p - 1 + i, 15 * p, p - 1)
    ]

    # Interpolate to obtain an approximating power series
    fp = R.lagrange_polynomial(points)
    v0 = v(fp[0])  # Valuation of constant coefficient
        
    # Compute lambda-invariant
    # Search for lambda up to 15
    # Large lambda values are very rare
    lambda_bound = 15
    j = 0
    while j <= lambda_bound:
        if v(fp[j]) <= 0: 
            break
        j += 1
    
    if j > lambda_bound:
        print(f'p={p}, twist={i}, Fundamental Discriminant={D} ==> lambda>={j}, v(const)={v0}')
        return fp, lambda_bound + 1, v0 
            
    slopes = []
    discstring = ''
    ramtype = [', unramified field extension', ', ramified field extension', ', trivial field extension']

    if j > 0:
        y=R.gen()
        for sh in shift:
            discstring = ''
            fpshift=fp(y+sh)
        
            fR = fpshift.truncate(j + 1) 
            try:       
                slopes = fR.newton_slopes()[0:j]  # Slopes of distinguished polynomial
            except Exception:
                slopes = 'NA'
        
            # Further analyze slopes with multiplicity >= 2
            # (Multiplicity 1 corresponds to a linear factor of the distinguished polynomial, hence a root in Zp)
            if j >= 2 and isinstance(slopes, list):
                myslopes = fR.newton_slopes(repetition=False)
                for myslope in myslopes:
                    # Case 1: Multiplicity 2 (Quadratic factor)
                    if slopes.count(myslope) == 2:
                        try:
                            g = fpshift.factor_of_slope(myslope)
                            Disc = g.discriminant()
                            vD = v(Disc)
                            if vD % 2 == 1:
                                rt = ramtype[1]  # Ramified quadratic extension
                            else:
                                # Unit part projected to GF(p)
                                u0 = Disc.unit_part().residue()  
                                if u0.is_square():
                                    rt = ramtype[2]  # Splits in Qp (two linear roots)
                                else:
                                    rt = ramtype[0]  # Unramified quadratic extension
                        
                            discstring += ', slope=' + str(myslope) + ' v(Disc)=' + str(vD) + rt
                        except Exception:
                            discstring += ', slope=' + str(myslope) + ' ***Error***'
                        
                    # Case 2: Multiplicity > 2 (Degree >= 3 factors, rare)
                    if slopes.count(myslope) > 2:  
                        try:
                            g = fpshift.factor_of_slope(myslope)
                            Disc = g.discriminant()
                            vD = v(Disc)                      
                            discstring += ', slope=' + str(myslope) + ' v(Disc)=' + str(vD)

                            # Fractional slopes correspond strictly to ramified extensions
                            if myslope not in ZZ:
                                discstring +=  ramtype[1] + ' of degree ' + str(slopes.count(myslope))
                            else:
                                s = ZZ(myslope)
                                for myfactor, multiplicity in g.factor():
                                    for _ in range(multiplicity):
                                        d = myfactor.degree()
                                        if d == 1:
                                            discstring += ', zero in Qp'
                                        else:
                                            is_unram = False
                                            is_ram = False

                                            X = myfactor.parent().gen()
                                            h = sum(myfactor[k] * (p**(k * s - d * s)) * X**k for k in range(d + 1))
                                        
                                            # Try Sage extension first
                                            try:
                                                E = K.extension(h, names='alpha')
                                                e_idx = E.ramification_index()
                                                if e_idx == 1:
                                                    is_unram = True
                                                elif e_idx > 1:
                                                    is_ram = True
                                            except Exception:
                                                # Fallback algebraic invariants
                                                v_fac_disc = v(myfactor.discriminant())
                                                v_norm = v_fac_disc - s * d * (d - 1)

                                                if v_fac_disc % 2 == 1:
                                                    is_ram = True  # Odd valuation is strictly ramified
                                                elif d == 2 and v_fac_disc % 2 == 0:
                                                    is_unram = True  # Even valuation for d=2 is strictly unramified
                                                elif v_norm == 0:
                                                    is_unram = True  # v_norm == 0 is strictly unramified
                                                else:
                                                    try:
                                                        hbar = h.change_ring(GF(p))
                                                        if hbar.is_irreducible():
                                                            is_unram = True
                                                    except Exception:
                                                        pass

                                            if is_unram:
                                                discstring += ramtype[0] + ' of degree ' + str(d)
                                            elif is_ram:
                                                discstring += ramtype[1] + ' of degree ' + str(d)
                        except Exception:
                            discstring += ', slope=' + str(myslope) + ' ***Error***'
            if j >= verb: # print line if lambda >= verb
                print(f'p={p}, twist={i}, Fundamental Discriminant={D} ==> lambda={j}, shift={sh}, v(const)={v0}, slopes={slopes}{discstring}')   
    else:  # j=0
        if verb==0:
            print(f'p={p}, twist={i}, Fundamental Discriminant={D} ==> lambda={j}, v(const)={v0}') 
    return fp, j, v0

In [2]:
shift = [0,3,6,9,12,15,18,21,24]
p=3 # prime
i=1 # Teichmuller twist
lamb=[0]*17
vals=[0]*15

for D in range(-2,-300,-1):
    if Integer(D).is_fundamental_discriminant()==False or gcd(D,p)!=1:        
        continue
    if i==1 and kronecker(D,p)==1:
        continue
    fp, j, v0 = power_series_shift(D,p,i,shift,1) 
    if j!= None:
        lamb[j]+=1
    if v0!= None:
        vals[v0]+=1

print(f'\np={p}, twist={i}, \ndistribution-lambda=      {lamb},\ndistribution-v(constant)= {vals}' )   

p=3, twist=1, Fundamental Discriminant=-31 ==> lambda=1, shift=0, v(const)=1, slopes=[1]
p=3, twist=1, Fundamental Discriminant=-31 ==> lambda=1, shift=3, v(const)=1, slopes=[1]
p=3, twist=1, Fundamental Discriminant=-31 ==> lambda=1, shift=6, v(const)=1, slopes=[2]
p=3, twist=1, Fundamental Discriminant=-31 ==> lambda=1, shift=9, v(const)=1, slopes=[1]
p=3, twist=1, Fundamental Discriminant=-31 ==> lambda=1, shift=12, v(const)=1, slopes=[1]
p=3, twist=1, Fundamental Discriminant=-31 ==> lambda=1, shift=15, v(const)=1, slopes=[3]
p=3, twist=1, Fundamental Discriminant=-31 ==> lambda=1, shift=18, v(const)=1, slopes=[1]
p=3, twist=1, Fundamental Discriminant=-31 ==> lambda=1, shift=21, v(const)=1, slopes=[1]
p=3, twist=1, Fundamental Discriminant=-31 ==> lambda=1, shift=24, v(const)=1, slopes=[2]
p=3, twist=1, Fundamental Discriminant=-139 ==> lambda=1, shift=0, v(const)=1, slopes=[1]
p=3, twist=1, Fundamental Discriminant=-139 ==> lambda=1, shift=3, v(const)=1, slopes=[4]
p=3, twist=1, 

In [3]:
shift=[0,5,10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100,105,110,115,120]
p=5 # prime
i=3 # Teichmuller twist
lamb=[0]*17
vals=[0]*15

for D in range(-2,-100,-1):
    if Integer(D).is_fundamental_discriminant()==False or gcd(D,p)!=1:        
        continue
    if i==1 and kronecker(D,p)==1:
        continue
    fp, j, v0 = power_series_shift(D,p,i,shift,1) 
    if j!= None:
        lamb[j]+=1
    if v0!= None:
        vals[v0]+=1

print(f'\np={p}, twist={i}, \ndistribution-lambda=      {lamb},\ndistribution-v(constant)= {vals}' )    

p=5, twist=3, Fundamental Discriminant=-83 ==> lambda=1, shift=0, v(const)=1, slopes=[1]
p=5, twist=3, Fundamental Discriminant=-83 ==> lambda=1, shift=5, v(const)=1, slopes=[2]
p=5, twist=3, Fundamental Discriminant=-83 ==> lambda=1, shift=10, v(const)=1, slopes=[1]
p=5, twist=3, Fundamental Discriminant=-83 ==> lambda=1, shift=15, v(const)=1, slopes=[1]
p=5, twist=3, Fundamental Discriminant=-83 ==> lambda=1, shift=20, v(const)=1, slopes=[1]
p=5, twist=3, Fundamental Discriminant=-83 ==> lambda=1, shift=25, v(const)=1, slopes=[1]
p=5, twist=3, Fundamental Discriminant=-83 ==> lambda=1, shift=30, v(const)=1, slopes=[2]
p=5, twist=3, Fundamental Discriminant=-83 ==> lambda=1, shift=35, v(const)=1, slopes=[1]
p=5, twist=3, Fundamental Discriminant=-83 ==> lambda=1, shift=40, v(const)=1, slopes=[1]
p=5, twist=3, Fundamental Discriminant=-83 ==> lambda=1, shift=45, v(const)=1, slopes=[1]
p=5, twist=3, Fundamental Discriminant=-83 ==> lambda=1, shift=50, v(const)=1, slopes=[1]
p=5, twist=3

In [4]:
shift=[7*k for k in range(49)]
p=7 # prime
i=4 # Teichmuller twist
lamb=[0]*17
vals=[0]*15

for D in range(2,100):
    if Integer(D).is_fundamental_discriminant()==False or gcd(D,p)!=1:        
        continue
    if i==1 and kronecker(D,p)==1:
        continue
    fp, j, v0 = power_series_shift(D,p,i,shift,1) 
    if j!= None:
        lamb[j]+=1
    if v0!= None:
        vals[v0]+=1

print(f'\np={p}, twist={i}, \ndistribution-lambda=      {lamb},\ndistribution-v(constant)= {vals}' )  

p=7, twist=4, Fundamental Discriminant=41 ==> lambda=1, shift=0, v(const)=1, slopes=[1]
p=7, twist=4, Fundamental Discriminant=41 ==> lambda=1, shift=7, v(const)=1, slopes=[1]
p=7, twist=4, Fundamental Discriminant=41 ==> lambda=1, shift=14, v(const)=1, slopes=[1]
p=7, twist=4, Fundamental Discriminant=41 ==> lambda=1, shift=21, v(const)=1, slopes=[1]
p=7, twist=4, Fundamental Discriminant=41 ==> lambda=1, shift=28, v(const)=1, slopes=[1]
p=7, twist=4, Fundamental Discriminant=41 ==> lambda=1, shift=35, v(const)=1, slopes=[1]
p=7, twist=4, Fundamental Discriminant=41 ==> lambda=1, shift=42, v(const)=1, slopes=[2]
p=7, twist=4, Fundamental Discriminant=41 ==> lambda=1, shift=49, v(const)=1, slopes=[1]
p=7, twist=4, Fundamental Discriminant=41 ==> lambda=1, shift=56, v(const)=1, slopes=[1]
p=7, twist=4, Fundamental Discriminant=41 ==> lambda=1, shift=63, v(const)=1, slopes=[1]
p=7, twist=4, Fundamental Discriminant=41 ==> lambda=1, shift=70, v(const)=1, slopes=[1]
p=7, twist=4, Fundament